In [1]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 11.2 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5316538 sha256=2c75b2994518817d42ce5d9647c936eed3ec10e53b507d6e58e845d1fe096d58
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


In [2]:
"""Global-memory and shared-memory adjacent differences with PyCUDA.

The operation is

    b[i] = a[i] - a[i-1],    i >= 1,
    b[0] = 0.

The example is intentionally simple.  Each input value is reused by at most two
neighboring output updates, so explicit shared-memory staging has only limited
reuse to exploit.  The script compares a direct global-memory kernel with a
shared-memory tiled kernel and reports CUDA-event timings over several powers
of two.

A CUDA-capable NVIDIA GPU, PyCUDA, and a working CUDA installation are required.
"""


'Global-memory and shared-memory adjacent differences with PyCUDA.\n\nThe operation is\n\n    b[i] = a[i] - a[i-1],    i >= 1,\n    b[0] = 0.\n\nThe example is intentionally simple.  Each input value is reused by at most two\nneighboring output updates, so explicit shared-memory staging has only limited\nreuse to exploit.  The script compares a direct global-memory kernel with a\nshared-memory tiled kernel and reports CUDA-event timings over several powers\nof two.\n\nA CUDA-capable NVIDIA GPU, PyCUDA, and a working CUDA installation are required.\n'

In [3]:
from __future__ import annotations

import argparse
from dataclasses import dataclass

import numpy as np

In [4]:
try:
    import pycuda.autoinit  # noqa: F401  # Creates the CUDA context.
    import pycuda.driver as cuda
    from pycuda.compiler import SourceModule
    import pycuda.gpuarray as gpuarray
except ImportError as exc:  # pragma: no cover - requires CUDA
    raise SystemExit(
        "PyCUDA and a working NVIDIA CUDA installation are required."
    ) from exc

In [5]:
CUDA_SOURCE = r"""
extern "C" {

__global__ void adjacent_global(
    const float * __restrict__ input,
    float * __restrict__ output,
    const int n)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    if (i == 0) {
        output[0] = 0.0f;
    } else {
        // Two direct, coalesced global-memory reads per output sample.
        output[i] = input[i] - input[i - 1];
    }
}

__global__ void adjacent_shared(
    const float * __restrict__ input,
    float * __restrict__ output,
    const int n)
{
    // One halo sample is required on the left of every block.
    extern __shared__ float tile[];

    const int tx = threadIdx.x;
    const int i = blockIdx.x * blockDim.x + tx;
    const int local = tx + 1;

    // Every thread stages one central value. Threads beyond n stage zero so
    // that all threads can still participate in __syncthreads().
    tile[local] = (i < n) ? input[i] : 0.0f;

    // The first thread in the block stages the one-element left halo.
    if (tx == 0) {
        const int halo = i - 1;
        tile[0] = (halo >= 0 && halo < n) ? input[halo] : 0.0f;
    }

    __syncthreads();

    if (i >= n) return;

    if (i == 0) {
        output[0] = 0.0f;
    } else {
        output[i] = tile[local] - tile[local - 1];
    }
}

} // extern "C"
"""

In [6]:
@dataclass
class TimingRow:
    n: int
    global_ms: float
    shared_ms: float
    speedup: float
    max_abs_difference: float


class AdjacentKernels:
    """Compile the two CUDA kernels once and reuse them for all grid sizes."""

    def __init__(self) -> None:
        module = SourceModule(CUDA_SOURCE, options=["-O3"], no_extern_c=True)
        self.global_kernel = module.get_function("adjacent_global")
        self.shared_kernel = module.get_function("adjacent_shared")

In [7]:
def cpu_reference(a: np.ndarray) -> np.ndarray:
    """Return the adjacent differences used to validate both GPU kernels."""
    out = np.zeros_like(a)
    out[1:] = a[1:] - a[:-1]
    return out

In [8]:
def _event_time_ms(launch, launches: int) -> float:
    """Measure an average kernel time using CUDA events.

    Several identical launches are enclosed between the same event pair.  This
    makes very short kernels easier to measure while keeping compilation,
    allocation, and host-device transfers outside the timed interval.
    """
    start = cuda.Event()
    stop = cuda.Event()
    start.record()
    for _ in range(launches):
        launch()
    stop.record()
    stop.synchronize()
    return float(start.time_till(stop)) / launches

In [9]:
def benchmark_size(
    n: int,
    *,
    block_size: int,
    warmup: int,
    repetitions: int,
    launches_per_sample: int,
    kernels: AdjacentKernels,
) -> TimingRow:
    """Validate and benchmark global and shared memory for one array size."""
    if n < 2:
        raise ValueError("n must be at least 2")
    if block_size <= 0:
        raise ValueError("block_size must be positive")

    # A deterministic nontrivial signal avoids a validation that succeeds only
    # for a linear ramp.
    x = np.arange(n, dtype=np.float32)
    a = (np.sin(0.0013 * x) + 0.25 * np.cos(0.0071 * x)).astype(np.float32)
    reference = cpu_reference(a)

    a_gpu = gpuarray.to_gpu(a)
    global_gpu = gpuarray.empty_like(a_gpu)
    shared_gpu = gpuarray.empty_like(a_gpu)

    grid_size = (n + block_size - 1) // block_size
    block = (block_size, 1, 1)
    grid = (grid_size, 1, 1)
    shared_bytes = (block_size + 1) * np.dtype(np.float32).itemsize
    n32 = np.int32(n)

    def launch_global() -> None:
        kernels.global_kernel(a_gpu, global_gpu, n32, block=block, grid=grid)

    def launch_shared() -> None:
        kernels.shared_kernel(
            a_gpu,
            shared_gpu,
            n32,
            block=block,
            grid=grid,
            shared=shared_bytes,
        )

    # Warm up JIT/caches and validate before measuring performance.
    for _ in range(warmup):
        launch_global()
        launch_shared()
    cuda.Context.synchronize()

    global_host = global_gpu.get()
    shared_host = shared_gpu.get()
    error_global = float(np.max(np.abs(global_host - reference)))
    error_shared = float(np.max(np.abs(shared_host - reference)))
    if error_global > 5e-6 or error_shared > 5e-6:
        raise RuntimeError(
            f"validation failed: global={error_global:.3e}, shared={error_shared:.3e}"
        )

    global_samples: list[float] = []
    shared_samples: list[float] = []

    # Alternate the order to reduce systematic bias from changing GPU clocks.
    for rep in range(repetitions):
        if rep % 2 == 0:
            global_samples.append(_event_time_ms(launch_global, launches_per_sample))
            shared_samples.append(_event_time_ms(launch_shared, launches_per_sample))
        else:
            shared_samples.append(_event_time_ms(launch_shared, launches_per_sample))
            global_samples.append(_event_time_ms(launch_global, launches_per_sample))

    global_ms = float(np.median(global_samples))
    shared_ms = float(np.median(shared_samples))
    speedup = global_ms / shared_ms

    return TimingRow(
        n=n,
        global_ms=global_ms,
        shared_ms=shared_ms,
        speedup=speedup,
        max_abs_difference=float(np.max(np.abs(global_host - shared_host))),
    )

In [10]:
def benchmark_scaling(
    *,
    min_power: int = 10,
    max_power: int = 24,
    block_size: int = 256,
    warmup: int = 3,
    repetitions: int = 7,
    launches_per_sample: int = 20,
) -> list[TimingRow]:
    """Benchmark powers of two and print a compact comparison table."""
    if max_power < min_power:
        raise ValueError("max_power must not be smaller than min_power")

    kernels = AdjacentKernels()
    rows: list[TimingRow] = []

    print("\nAdjacent differences: direct global memory vs explicit shared memory")
    print("speedup = T_global / T_shared; values > 1 favor shared memory\n")
    print(
        f"{'N':>10s} {'global [ms]':>14s} {'shared [ms]':>14s} "
        f"{'speedup':>10s} {'|G-S|max':>12s}"
    )
    print("-" * 66)

    for power in range(min_power, max_power + 1):
        row = benchmark_size(
            2**power,
            block_size=block_size,
            warmup=warmup,
            repetitions=repetitions,
            launches_per_sample=launches_per_sample,
            kernels=kernels,
        )
        rows.append(row)
        print(
            f"{row.n:10d} {row.global_ms:14.6f} {row.shared_ms:14.6f} "
            f"{row.speedup:10.4f} {row.max_abs_difference:12.3e}"
        )

    print(
        "\nInterpretation: each input value participates in only two neighboring "
        "differences. The shared-memory kernel roughly halves explicit global "
        "loads, but must also stage a tile, load one halo per block, execute a "
        "barrier, and perform shared-memory accesses. Modern caches can already "
        "capture much of this tiny reuse, so a speedup is not guaranteed."
    )
    return rows

In [11]:
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--min-power", type=int, default=10)
    parser.add_argument("--max-power", type=int, default=24)
    parser.add_argument("--block-size", type=int, default=256)
    parser.add_argument("--warmup", type=int, default=3)
    parser.add_argument("--repetitions", type=int, default=7)
    parser.add_argument("--launches", type=int, default=20)
    return parser

In [12]:
def main(argv=None) -> int:
    args = build_parser().parse_args(argv)
    benchmark_scaling(
        min_power=args.min_power,
        max_power=args.max_power,
        block_size=args.block_size,
        warmup=args.warmup,
        repetitions=args.repetitions,
        launches_per_sample=args.launches,
    )
    return 0


## GPU used for the benchmark

The global/shared-memory crossover is hardware dependent. Record the assigned GPU before running the timing campaign.


In [13]:
device = cuda.Device(0)
print("GPU:", device.name())
print("Compute capability:", device.compute_capability())
print(f"Global memory: {device.total_memory() / 1024**3:.2f} GiB")
print(
    "Shared memory per block:",
    f"{device.get_attribute(cuda.device_attribute.MAX_SHARED_MEMORY_PER_BLOCK) / 1024:.1f} KiB",
)


GPU: Tesla T4
Compute capability: (7, 5)
Global memory: 14.56 GiB
Shared memory per block: 48.0 KiB


## Run the benchmark in Colab

Colab/Jupyter adds its own command-line option `-f <kernel.json>`. Calling `main([...])` explicitly prevents that internal option from being passed to `argparse`.


In [14]:
main([
    "--min-power", "10",
    "--max-power", "24",
    "--block-size", "256",
    "--warmup", "3",
    "--repetitions", "7",
    "--launches", "20",
])



Adjacent differences: direct global memory vs explicit shared memory
speedup = T_global / T_shared; values > 1 favor shared memory

         N    global [ms]    shared [ms]    speedup     |G-S|max
------------------------------------------------------------------
      1024       0.018976       0.020933     0.9065    0.000e+00
      2048       0.020474       0.021382     0.9575    0.000e+00
      4096       0.022171       0.021094     1.0510    0.000e+00
      8192       0.019843       0.020480     0.9689    0.000e+00
     16384       0.020632       0.020704     0.9965    0.000e+00
     32768       0.019699       0.020688     0.9522    0.000e+00
     65536       0.021490       0.021275     1.0101    0.000e+00
    131072       0.019371       0.020792     0.9317    0.000e+00
    262144       0.020070       0.022232     0.9028    0.000e+00
    524288       0.023758       0.024698     0.9620    0.000e+00
   1048576       0.043418       0.055994     0.7754    0.000e+00
   2097152       0.0

0